# Convert global country mask into SHERPA resolution EU country mask

Step 1: crop global mask into EU bounds   
Step 2: check how many cells "overlap", i.e. are there many cells where multiple countries claim that cell? If there are only a small number, continue (if a large number then a different method will be needed)  
Step 3: collapes the (country, lat, lon) format into (lat, lon) with country numbers as index (argmax chooses the first country in any overlapping cells) so only one country per cell is assigned  
Step 4: interpolate to SHERPA grid  
Step 5: expand (lat, lon) to (country, lat, lon)

In [1]:
import os
import xarray as xr
import numpy as np
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Path config ===
EU_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "processed")
MASKS_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country", "country masks")

In [3]:
# Load one SHERPA file for lat/lon contraints and resolution
eu_file = "EU_concentration_H_2040.nc"
eu_path = os.path.join(EU_DIR, eu_file)
eu = xr.open_dataarray(eu_path)

# Load country mask
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
mask = xr.open_dataarray(mask_path)

# Crop mask to EU bounding box (with buffer for edge safety)
lat_min, lat_max = float(eu.latitude.min()), float(eu.latitude.max())
lon_min, lon_max = float(eu.longitude.min()), float(eu.longitude.max())
buffer = 0.5

mask_eu = mask.sel(
    lat=slice(lat_min - buffer, lat_max + buffer),
    lon=slice(lon_min - buffer, lon_max + buffer),
)

n_multi_eu = int((mask_eu.sum(dim="country") > 1).sum())
if n_multi_eu > 0:
    print(f"WARNING: {n_multi_eu} cells in the EU bbox claimed by >1 country. "
          "Review before proceeding -- argmax below will silently pick the "
          "first match in the country dimension, which may not be what you want.")

print("REVIEW: only 10 cells affected, keep argmax to pick first match")

REVIEW: only 10 cells affected, keep argmax to pick first match


In [22]:
# Collapse (country, lat, lon) from masks -> single country-index grid
# argmax gives the index of the first country with mask==1 in each cell.
# Cells with no country (all zeros, e.g. ocean) will incorrectly get
# index 0 -- so mask those out explicitly using a coverage count.
country_idx = mask_eu.argmax(dim="country")  # int, shape (lat, lon) at 0.1x0.1
has_country = mask_eu.sum(dim="country") > 0
country_idx = country_idx.where(has_country)  # NaN where no country

# Nearest-neighbor regrid the country-index grid onto EU coords
# This is the only "resampling" step, and it's nearest-neighbor because
# country ID is categorical.
country_idx_eu_res = country_idx.interp(
    lat=eu.latitude, lon=eu.longitude, method="nearest"
)

description = ("DataArray that contains a gridded mask for all"
               " countries in the SHERPA EU bounds (lat, lon)")
country_idx_eu_res.attrs["description"] = description

out_file = "GBD_Country_Masks_EU_SHERPA_res_latlon.nc"
out_path = os.path.join(MASKS_DIR, out_file)
country_idx_eu_res.to_netcdf(out_path)

print("All processing complete.")

All processing complete.


In [5]:
def expand_country_idx_to_mask(country_idx, country_names):
    """
    country_idx: (lat, lon) DataArray of integer country codes (NaN = no country)
    country_names: ordered list/array where country_names[i] is the name for code i

    Returns: (country, lat, lon) DataArray, 1 where that country's code matches, else 0
    """
    n_countries = len(country_names)

    # Broadcast comparison: (country, lat, lon) == country_idx (lat, lon)
    country_code = xr.DataArray(np.arange(n_countries), dims="country",
                                coords={"country": country_names})

    expanded = (country_idx == country_code).astype(np.uint8)

    # Cells that were NaN in country_idx (no country) should be all-zero,
    # see cell above "country_idx = country_idx.where(has_country)"
    has_country = ~country_idx.isnull()
    expanded = expanded.where(has_country, other=0)

    return expanded

In [20]:
country_list = mask.country.values

country_mask_expanded = expand_country_idx_to_mask(country_idx_eu_res, country_list)

description = ("DataArray that contains gridded masks for all"
               " countries in the SHERPA EU bounds (country, lat, lon)")
country_mask_expanded.attrs["description"] = description

out_file = "GBD_Country_Masks_EU_SHERPA_res.nc"
out_path = os.path.join(MASKS_DIR, out_file)
country_mask_expanded.to_netcdf(out_path)

print("All processing complete.")

All processing complete.
